# 第09课：Steam 字符串清洗

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用同目录的 [steam.csv](steam.csv)。课后独立练习见 [chapter09_Steam字符串清洗_课后练习.ipynb](chapter09_Steam字符串清洗_课后练习.ipynb)，数据为 [steam_sample.csv](steam_sample.csv)。

## 学习目标

1. 换一份 Steam 表时，先打印 header 与两行样例，根据列名确认下标，不沿用移动游戏表的下标。
2. 使用 `str.replace` 去掉空格、脏词或逗号；并说明字符串不可变，必须把结果赋回变量。
3. 用 `capitalize`（或同样规则）统一大小写，使 `windows` 与 `Windows` 能合并。
4. 把看起来像数的字段去杂质后再 `int`；能解释未清洗时的 `ValueError`。
5. 用循环清洗整列：同一列清洗前后各打印 3 行；统计评价数转 int 成功的条数。

## 学习知识点

| 认列 | 字符串清洗 | 转成能算的值 |
| --- | --- | --- |
| 先打印 header | `replace` 去空格 / 脏词 | 去逗号再 `int` |
| 不套用旧下标 | 必须赋回变量 | 杂质会 `ValueError` |
| 写下标常量 | `capitalize` 统一大小写 | 循环洗整列 |

## 基础回顾与案例提问

第 3–8 课一直用移动游戏表。那里 `title=1`、`score=2`、`isAndroid=6`。现在换 [steam.csv](steam.csv)，约 2.7 万行，字段更脏、列更多。**下标必须重新认。** 第 0 行样例：`Counter-Strike`，发行日 `2000-11-01`，平台 `windows;mac;linux`，好评 `124534`，拥有者区间 `10000000-20000000`。第 1 行：`Team Fortress Classic`，`1999-04-01`，同样三个平台，好评 `3318`，拥有者 `5000000-10000000`。

本课不要把 2.7 万行整表打印出来：案例只打印 header 和两行样例，循环时也不要 `print` 每一行。

1. **R.1** 移动游戏表下标 2 是评分。Steam 表不看 header 也写 `row[2]`，取到的还是评分吗？
2. **R.2** `text = "windows"` 之后执行 `text.replace("w", "W")`，不赋回变量，再 `print(text)`，会看到什么？
3. **R.3** `int("12,4534")` 会怎样？先去掉逗号再 `int`，得到的是什么？

**作答：** 预测：____；依据：____；验证后说明：____。

使用 Python 3；只需标准库 `csv`。从该文件夹启动内核。打开 CSV 使用 `encoding="utf-8-sig"`。本课不讲正则，不展开 `split` 的全部用法（平台字段按 `;` 切开即可）。不要使用列表推导式、pandas 或 `re`。综合练习的整列清洗对照和 int 成功条数必须由你的代码算出，题面不提供这些验收数字。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


## 1. 换表先认列

### 理论知识

**换一份表，第一件事不是写公式，是看表头。** `csv.reader` 读进来以后，`header = rows[0]` 是列名，`body = rows[1:]` 才是数据。列下标从 0 开始，必须用当前这份 header 核对。

移动游戏表只有 7 列；Steam 表有 18 列。即使某个下标数字碰巧相同，含义也可能完全不同：移动游戏的 `row[2]` 是评分，Steam 的 `row[2]` 是发行日期。同目录的 [mobile_game_info.csv](mobile_game_info.csv) 只用来对照，不要把那套下标抄过来。

这份 `steam.csv` 很大。认列时打印 header 和两行样例就够了，**不要把整张表 dump 到屏幕上**。

### 案例：打印 header 和两行样例


In [1]:
import csv

with open("steam.csv", "r", encoding="utf-8-sig", newline="") as file:
    rows = list(csv.reader(file))
header = rows[0]
body = rows[1:]
print("Column names:", header)
print("Header length:", len(header))
print("Sample row 0:", body[0][1], body[0][2], body[0][6], body[0][12], body[0][16])
print("Sample row 1:", body[1][1], body[1][2], body[1][6], body[1][12], body[1][16])

with open("mobile_game_info.csv", "r", encoding="utf-8-sig", newline="") as file:
    mobile_header = list(csv.reader(file))[0]
print("Mobile header (do not reuse):", mobile_header)


Column names: ['appid', 'name', 'release_date', 'english', 'developer', 'publisher', 'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags', 'achievements', 'positive_ratings', 'negative_ratings', 'average_playtime', 'median_playtime', 'owners', 'price']
Header length: 18
Sample row 0: Counter-Strike 2000-11-01 windows;mac;linux 124534 10000000-20000000
Sample row 1: Team Fortress Classic 1999-04-01 windows;mac;linux 3318 5000000-10000000
Mobile header (do not reuse): ['id', 'title', 'score', 'game_type', 'pulisher', 'download', 'isAndroid']


### 讲解

Steam 的列名是 `appid, name, release_date, ...`。第一行游戏是 Counter-Strike，第二行是 Team Fortress Classic。移动游戏表头仍是 `id, title, score, ...`。两套名字对不上，下标就不能共用。

本格没有打印 `len(body)` 的验收数字，也没有列出全部数据行。整表有多少行、整列清洗后什么样，留给 P1 你自己算。

### 易错点与练习

打开失败时先检查工作目录和文件名。中文或首列名异常时检查是否漏了 `utf-8-sig`。不要把表头送进后面的清洗循环。

1. **K1.1** 根据刚打印的 header，写出 `name`、`release_date`、`platforms` 的从 0 开始的下标。
2. **K1.2** 若把移动游戏的 `row[2]`（评分）习惯套到 Steam 上，实际取到的是哪一列？

**作答：** 三个下标：____；误用下标 2 会取到：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. 把本课要用的列写成常量

### 理论知识

认完列之后，把分析要用的下标写成常量，后面就不要再猜数字。本课先锁定：

- `name = 1`
- `release_date = 2`
- `developer = 4`
- `platforms = 6`
- `positive_ratings = 12`
- `owners = 16`

常量仍然要和 `header.index("列名")` 对得上。对得上，说明认列成功；对不上，停下来，不要继续洗。

### 案例：核对六列下标


In [2]:
NAME = 1
RELEASE_DATE = 2
DEVELOPER = 4
PLATFORMS = 6
POSITIVE_RATINGS = 12
OWNERS = 16
print("name index:", header.index("name"), "constant:", NAME)
print("release_date index:", header.index("release_date"), "constant:", RELEASE_DATE)
print("developer index:", header.index("developer"), "constant:", DEVELOPER)
print("platforms index:", header.index("platforms"), "constant:", PLATFORMS)
print("positive_ratings index:", header.index("positive_ratings"), "constant:", POSITIVE_RATINGS)
print("owners index:", header.index("owners"), "constant:", OWNERS)
print("Row0 fields:", body[0][NAME], body[0][RELEASE_DATE], body[0][DEVELOPER], body[0][PLATFORMS], body[0][POSITIVE_RATINGS], body[0][OWNERS])


name index: 1 constant: 1
release_date index: 2 constant: 2
developer index: 4 constant: 4
platforms index: 6 constant: 6
positive_ratings index: 12 constant: 12
owners index: 16 constant: 16
Row0 fields: Counter-Strike 2000-11-01 Valve windows;mac;linux 124534 10000000-20000000


### 讲解

六个 `header.index` 应分别等于 1、2、4、6、12、16。第一行开发者是 Valve，平台是 `windows;mac;linux`，好评文本是 `124534`，拥有者是 `10000000-20000000`。这些值接下来会被清洗或转换。

课后的 `steam_sample.csv` 列名相同，但仍要自己对着 header 认一遍，不要只背数字。

### 易错点与练习

`platforms` 在下标 6，移动游戏的 `isAndroid` 也在下标 6。数字碰巧一样，含义完全不是一回事。

1. **K2.1** `positive_ratings` 为什么不是下标 2？2 号列叫什么？
2. **K2.2** 把 `NAME` 误写成 0，第一行会打印出什么字段？

**作答：** 下标 2 的列名：____；NAME=0 会取到：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. `replace`：去掉空格和脏词

### 理论知识

**`replace(旧, 新)` 返回一份替换后的新字符串。** 本课先用它做两件小事：去掉空格，把不统一的词改成统一写法。

分类文本常常写成 `Action; FPS ;action`：中间有空格，同一个词忽大忽小。若直接拿去 `==` 或当字典的键，`Action` 和 `action` 会被当成两类。

本课不讲正则。能用 `replace` 去掉的杂质，就用 `replace`。

### 案例：清洗一小段脏分类


In [3]:
dirty = "Action; FPS ;action"
step1 = dirty.replace(" ", "")
step2 = step1.replace("action", "Action")
print("Original:", dirty)
print("Remove spaces:", step1)
print("Unify action:", step2)


Original: Action; FPS ;action
Remove spaces: Action;FPS;action
Unify action: Action;FPS;Action


### 讲解

先去空格得到 `Action;FPS;action`，再把 `action` 换成 `Action`，得到 `Action;FPS;Action`。顺序有时有影响：如果先替换单词再去空格，结果仍应检查一遍。

`replace` 会替换所有匹配到的片段。本格只处理这一小段样例，不是整张 Steam 表。

### 易错点与练习

`replace` 区分大小写：`action` 和 `Action` 不是同一段文字。

1. **K3.1** `"Action; FPS ;action".replace(" ", "")` 之后，分号两边还有空格吗？
2. **K3.2** 若只执行 `dirty.replace("action", "Action")`、不去空格，中间的 ` FPS ` 还在吗？

**作答：** 去空格后的串：____；只改单词时：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. 字符串不可变：必须赋回变量

### 理论知识

**字符串改不了“原来那一份”，只能得到新的一份。** `text.replace(...)` 本身不会改掉 `text`。若需要后面用新内容，必须写：

```python
text = text.replace(" ", "")
```

只调用、不赋值，后面的 `print(text)` 仍是旧串。这是本课最常见的清洗故障。

故障写法只放在文字里，不要放进最终运行顺序：

```python
# Deliberate fault: replace without assigning back.
text = "windows"
text.replace("windows", "Windows")
print(text)
```

### 案例：赋回之后才看得到清洗结果


In [4]:
text = "windows"
ignored = text.replace("windows", "Windows")
print("Original after unassigned replace:", text)
print("Returned new string:", ignored)
text = text.replace("windows", "Windows")
print("After assigning back:", text)


Original after unassigned replace: windows
Returned new string: Windows
After assigning back: Windows


### 讲解

第一次 `replace` 的返回值进了 `ignored`，`text` 仍是 `windows`。第二次把返回值赋回 `text`，才变成 `Windows`。清洗平台、去逗号，都要遵守同一条：用新值覆盖旧变量，或写入新列表的新格子。

### 易错点与练习

1. **K4.1** 预测故障片段 `print(text)` 的结果。它会不会变成 `Windows`？
2. **K4.2** 清洗评价数时写了 `row[12].replace(",", "")` 却不保存，下一行 `int(row[12])` 用的是哪一份文字？

**作答：** 未赋回的 print：____；未保存就 int：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. `capitalize`：统一 windows 与 Windows

### 理论知识

**大小写不同，对 Python 来说就是不同的字符串。** `"windows" == "Windows"` 为假。做平台频数时，不统一大小写就会得到两个键，实际却是同一类平台。

`capitalize()` 把第一位变成大写、其余变成小写：`"windows"` → `"Windows"`，`"WINDOWS"` → `"Windows"`，`"Windows"` 仍是 `"Windows"`。平台字段里若有多段，需要先按 `;` 切开，再对每一段 `strip` 和 `capitalize`，最后用 `;` 拼回去。`split` 在本课只用于“按分号切开”，不展开其他用法。

### 案例：未统一时两个键，统一后合成一个


In [5]:
raw_list = ["windows", "Windows", "windows", "LINUX"]
print("windows == Windows:", "windows" == "Windows")
freq_raw = {}
for item in raw_list:
    if item not in freq_raw:
        freq_raw[item] = 0
    freq_raw[item] = freq_raw[item] + 1
print("Before unify:", freq_raw)

freq_clean = {}
for item in raw_list:
    key = item.strip().capitalize()
    if key not in freq_clean:
        freq_clean[key] = 0
    freq_clean[key] = freq_clean[key] + 1
print("After capitalize:", freq_clean)
print("windows ->", "windows".capitalize(), "; LINUX ->", "LINUX".capitalize())


windows == Windows: False
Before unify: {'windows': 2, 'Windows': 1, 'LINUX': 1}
After capitalize: {'Windows': 3, 'Linux': 1}
windows -> Windows ; LINUX -> Linux


### 讲解

未清洗时 `windows` 和 `Windows` 各计一次以上；`LINUX` 也不会和 `Linux` 合并。`capitalize` 之后键变成 `Windows` 与 `Linux`。这就是“为了让 `==` 或字典键稳定”而洗大小写。

Steam 原表里的平台多为全小写 `windows;mac;linux`。洗完应变为 `Windows;Mac;Linux`。不要假设课后样本已经帮你洗好。

### 易错点与练习

`capitalize` 只照顾整段的第一位。`"mac;linux"` 整段调用会得到 `"Mac;linux"`，linux 不会被改掉。所以要先切开再逐段处理。

1. **K5.1** `"windows;mac".capitalize()` 的结果是什么？为什么不够？
2. **K5.2** 统一大小写前后，频数的 keys 可能差在哪里？

**作答：** 整段 capitalize：____；keys 的差别：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. 循环清洗平台列

### 理论知识

洗一格和洗一列不是一回事。正确顺序：写出处理**一个**平台字符串的步骤，再 `for` 循环整列。本课用新列表收集清洗结果，不在遍历时改乱旧表。

对一格：`split(";")` → 每段 `strip().capitalize()` → `";".join(...)`。对整列：准备 `cleaned_platforms = []`，循环 `append`。验收时同一列清洗前后各打印 3 行，而不是打印 2.7 万行。

### 案例：第一行平台清洗前后，以及前三行对照


In [6]:
def clean_platforms(text):
    parts = text.split(";")
    cleaned_parts = []
    for part in parts:
        cleaned_parts.append(part.strip().capitalize())
    return ";".join(cleaned_parts)

before = body[0][PLATFORMS]
after = clean_platforms(before)
print("First row platforms before:", before)
print("First row platforms after:", after)

print("First 3 rows before / after:")
index = 0
for row in body[:3]:
    old = row[PLATFORMS]
    new = clean_platforms(old)
    print(index, row[NAME], old, "->", new)
    index = index + 1


First row platforms before: windows;mac;linux
First row platforms after: Windows;Mac;Linux
First 3 rows before / after:
0 Counter-Strike windows;mac;linux -> Windows;Mac;Linux
1 Team Fortress Classic windows;mac;linux -> Windows;Mac;Linux
2 Day of Defeat windows;mac;linux -> Windows;Mac;Linux


### 讲解

第一行 `windows;mac;linux` 变成 `Windows;Mac;Linux`。前三行的对照已经满足“看清洗有没有生效”。把整列都 `append` 进新列表，是 P1 的事；本格不要把全表平台打印出来。

`clean_platforms` 下一课还可以再用。函数体里是普通的 `for` + `append`，没有正则。

### 易错点与练习

只洗了屏幕上看到的第一行，并不等于整列都洗了。P1 要用循环覆盖全部数据行，再抽 3 行核对。

1. **K6.1** 若 `split` 之后忘记 `strip`，`" mac"` 经 `capitalize` 会变成什么？
2. **K6.2** 验收为什么要求前后各打印 3 行，而不是 `print(body)`？

**作答：** 带空格的一段：____；不 dump 整表的理由：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. 去逗号再 `int`

### 理论知识

**看起来像数，并不就是 `int`。** 带逗号的 `"12,4534"` 不能直接转换：

```python
# Deliberate fault: int() on a comma-separated digit string.
int("12,4534")
```

上面会 `ValueError`。正确顺序：先 `replace(",", "")`，必要时再去掉空格，然后 `int(...)`。Steam 表里的 `positive_ratings` 有的已经是纯数字文本；仍然要养成“先去杂质再转换”的写法，换脏数据时才不会翻车。

不要在最终运行的格子里保留会报错的 `int("12,4534")`。故障只写在 Markdown。

### 案例：去掉逗号后转换；再转前 5 行好评


In [7]:
dirty_number = "12,4534"
cleaned_number = dirty_number.replace(",", "")
print("Dirty text:", dirty_number)
print("After replace comma:", cleaned_number)
print("int after cleaning:", int(cleaned_number))

print("First 5 positive_ratings as int:")
converted = 0
for row in body[:5]:
    text = row[POSITIVE_RATINGS].replace(",", "").replace(" ", "")
    value = int(text)
    converted = converted + 1
    print(row[NAME], row[POSITIVE_RATINGS], "->", value)
print("Converted among first 5:", converted)


Dirty text: 12,4534
After replace comma: 124534
int after cleaning: 124534
First 5 positive_ratings as int:
Counter-Strike 124534 -> 124534
Team Fortress Classic 3318 -> 3318
Day of Defeat 3416 -> 3416
Deathmatch Classic 1273 -> 1273
Half-Life: Opposing Force 5250 -> 5250
Converted among first 5: 5


### 讲解

`"12,4534"` 去掉逗号后是 `"124534"`，`int` 得到 124534。这和 Counter-Strike 的好评数字相同，只是中间多了一个教学用的逗号。

本格只转换前 5 行，用来确认写法。全表成功条数是 P1 验收项，不要在讲义里填写，也不要写死。

### 易错点与练习

`int("12,4534")` 的报错不是因为数字太大，而是因为逗号不是数字字符。去掉逗号后再转。

1. **K7.1** 为什么故障 `int("12,4534")` 会 `ValueError`？修正表达式怎么写？
2. **K7.2** 本格 `converted` 统计的是前 5 行还是全表？全表成功条数应该在哪一格算？

**作答：** ValueError 原因与修正：____；converted 的范围：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 整列转换的写法；owners 区间先看一眼

### 理论知识

评价数要能留给第 10 课做档位，必须先变成整数。写法仍是循环 + 计数：成功则 `converted + 1`。本课案例继续只用前几行演示结构；P1 才对 `body` 全表计数。

`owners` 是区间文本，例如 `"10000000-20000000"`。入门做法：去掉逗号和空格，按 `-` 切成两段，分别 `int`，得到 `(low, high)`。复杂解析的完整练习放在课后 P3，当堂只要能看懂第一行。

### 案例：前 5 行转换结构；解析第一行 owners


In [8]:
def parse_int_field(text):
    cleaned = text.replace(",", "").replace(" ", "")
    return int(cleaned)

converted = 0
failed = 0
for row in body[:5]:
    parse_int_field(row[POSITIVE_RATINGS])
    converted = converted + 1
print("First 5 int conversions succeeded:", converted, "failed:", failed)

owners_text = body[0][OWNERS]
cleaned_owners = owners_text.replace(",", "").replace(" ", "")
owner_parts = cleaned_owners.split("-")
low = int(owner_parts[0])
high = int(owner_parts[1])
print("owners text:", owners_text)
print("owners low/high:", low, high)


First 5 int conversions succeeded: 5 failed: 0
owners text: 10000000-20000000
owners low/high: 10000000 20000000


### 讲解

前 5 行转换用来确认函数能跑。`failed` 在本样例里是 0，不代表你已经验收了全表。第一行 owners 解析为 10000000 与 20000000 两个整数。课后样本仍是同一列名，但只含 50 条数据，必须重新读取，不能沿用本笔记本的 `body`。

### 易错点与练习

`split("-")` 只在本格当“切开区间”用。不要在这里展开正则或更多 split 参数。

1. **K8.1** 若 `owners` 写成 `"10,000,000-20,000,000"`，不先 `replace(",", "")` 能 `int` 吗？
2. **K8.2** P1 的 int 成功条数，循环应走 `body[:5]` 还是全部 `body`？

**作答：** 带逗号的区间：____；P1 循环范围：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：独立完成平台清洗与评价数转换

使用同一份 [steam.csv](steam.csv)。先认列，再清洗。不要把整表打印出来。不要把 int 成功条数写死在代码或题面答案里——必须打印计算值。

当堂验收：同一列清洗前后各 print 3 行；评价数转 int 的成功计数。

### P1.1　读取与认列

独立写出读取代码，保留 `rows`、`header`、`body`。打印 header、两行样例（建议只打印 name / platforms / positive_ratings / owners），并用 `header.index` 说明 `platforms` 与 `positive_ratings` 的下标。不要 dump 整表。


In [ ]:
# P1.1: Read and inspect the Steam header and two sample rows here.


### P1.2　平台列清洗前后对照

写出平台清洗（切开、`strip`、`capitalize`、拼回）。对整列循环，把结果收进新列表。打印同一列清洗前后各 3 行（含游戏名，便于核对）。

**作答：** platforms 下标依据：____；前 3 行清洗前后：____。


In [ ]:
# P1.2: Clean the platforms column and print 3 rows before / after here.


### P1.3　评价数转 int 与成功计数

对 `positive_ratings` 去逗号（若有）再 `int`。循环全部数据行，统计成功条数；不要只转 5 行就当作验收完成。打印成功计数。重启内核，从头运行已完成的代码。故障片段留在文字中。

**作答：** 转换前是否去逗号：____；成功条数如何得到：____；重启后的验证情况：____。


In [ ]:
# P1.3: Convert positive_ratings to int and print the success count here.


## 本章总结

1. 换表先认列：Steam 的 `release_date` 在下标 2，不是移动游戏的评分。
2. `replace` 得到新字符串，必须赋回变量。
3. `capitalize` 用来合并 `windows` 与 `Windows`；多段平台要先按 `;` 切开。
4. `int("12,4534")` 会 `ValueError`；先去逗号再转换。
5. 验收看清洗前后 3 行对照，以及评价数转 int 的成功计数；不要 dump 整表。

保留列常量、平台清洗和评价数转换。第 10 课要用发行年和整好的好评数字做切片与摘要。

课后请打开 [chapter09_Steam字符串清洗_课后练习.ipynb](chapter09_Steam字符串清洗_课后练习.ipynb)，使用 [steam_sample.csv](steam_sample.csv) 独立完成 P1、P2（P3 选做）。课后只有 50 条数据，必须重新对着 header 认列，不能把课堂笔记本里的 `body` 直接拿去用。
